In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RAW_PATH = "../data/raw/jigsaw/train.csv"
OUT_DIR = "../data/processed"
SEED = 42

# 9 identities used in the official Jigsaw evaluation
MAIN_IDENTITIES = [
    "male", "female", "homosexual_gay_or_lesbian", "christian", "jewish",
    "muslim", "black", "white", "psychiatric_or_mental_illness",
]

# 1. Load Data

We load the raw Jigsaw training file and create the binary label (toxic when `target >= 0.5`).

In [2]:
df = pd.read_csv(RAW_PATH)
df["label"] = (df["target"] >= 0.5).astype(int)

print(f"Rows: {len(df):,}")
print(f"Toxic: {df['label'].mean() * 100:.2f}%")

Rows: 1,804,874
Toxic: 8.00%


# 2. Cleaning

We apply the decisions from the EDA:
1.  remove empty texts and comments without letters 
2.  keep one copy of each repeated text.
3.  when the copies of a text disagree on the label, the kept copy takes the mean `target` of all copies.

In [3]:
n_start = len(df)

# empty
df = df[df["comment_text"].notna()]
print(f"Removed empty texts: {n_start - len(df):,}")

#comments without any letter 
n_before = len(df)
df = df[df["comment_text"].str.contains(r"[A-Za-z]", regex=True)]
print(f"Removed comments without letters: {n_before - len(df):,}")

Removed empty texts: 3
Removed comments without letters: 368


In [4]:
# nomalize text
df["text_norm"] = df["comment_text"].str.lower().str.split().str.join(" ")

# take random sample
df = df.sample(frac=1, random_state=SEED)

groups = df.groupby("text_norm")
conflict = groups["label"].transform("nunique") > 1

# contradictory labels take means
df.loc[conflict, "target"] = groups["target"].transform("mean")[conflict]
df["label"] = (df["target"] >= 0.5).astype(int)

n_before = len(df)
df = df.drop_duplicates("text_norm", keep="first")

print(f"Texts with contradictory labels: {df.loc[conflict.loc[df.index], 'text_norm'].nunique():,}")
print(f"Removed duplicate copies: {n_before - len(df):,}")
print(f"\nRows after cleaning: {len(df):,} ({n_start - len(df):,} removed in total)")
print(f"Toxic after cleaning: {df['label'].mean() * 100:.2f}%")

Texts with contradictory labels: 808
Removed duplicate copies: 28,154

Rows after cleaning: 1,776,349 (28,525 removed in total)
Toxic after cleaning: 7.98%


In [5]:
columns = ["id", "comment_text", "target", "label", "article_id"] + MAIN_IDENTITIES + ["text_norm"]
df = df[columns].sort_values("id").reset_index(drop=True)
df.head()

,id,comment_text,target,label,article_id,male,female,homosexual_gay_or_lesbian,christian,jewish,muslim,black,white,psychiatric_or_mental_illness,text_norm
0,59848,"This is so cool. It's like, 'would you want yo...",0.000000,0,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"this is so cool. it's like, 'would you want yo..."
1,59849,Thank you!! This would make my life a lot less...,0.000000,0,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,thank you!! this would make my life a lot less...
2,59852,This is such an urgent design problem; kudos t...,0.000000,0,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,this is such an urgent design problem; kudos t...
3,59855,Is this something I'll be able to install on m...,0.000000,0,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,is this something i'll be able to install on m...
4,59856,haha you guys are a bunch of losers.,0.893617,1,2006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,haha you guys are a bunch of losers.


# 3. Train, Validation and Test Split

We split the data 80/10/10 with stratific division:

1. all comments of an article go to the same set (grouped by `article_id`)
2. toxic rate is kept similar in every set.

The data is divided into 10 folds: fold 0 is the test set, fold 1 the validation set and the remaining 8 the training set.

In [6]:
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

fold = np.empty(len(df), dtype=int)
for k, (_, fold_idx) in enumerate(sgkf.split(df, df["label"], groups=df["article_id"])):
    fold[fold_idx] = k

df["split"] = np.select([fold == 0, fold == 1], ["test", "validation"], default="train")
df["split"].value_counts()

split
train         1421077
validation     177637
test           177635
Name: count, dtype: int64

# 4. Checks and Saving

We check :
1. the size and toxic rate of each set
2.  that no article and no text appears in more than one set
3. how many comments of each official subgroup each set contains.

In [7]:
order = ["train", "validation", "test"]
summary = df.groupby("split").agg(comments=("label", "size"), toxic=("label", "mean")).loc[order]
summary["share (%)"] = summary["comments"] / len(df) * 100
summary["toxic (%)"] = summary.pop("toxic") * 100
summary.round(2)

,comments,share (%),toxic (%)
split,,,
train,1421077,80.0,7.98
validation,177637,10.0,7.99
test,177635,10.0,7.98


In [8]:
articles_in_several_sets = (df.groupby("article_id")["split"].nunique() > 1).sum()
texts_in_several_sets = (df.groupby("text_norm")["split"].nunique() > 1).sum()

print(f"Articles in more than one set: {articles_in_several_sets}")
print(f"Texts in more than one set: {texts_in_several_sets}")

Articles in more than one set: 0
Texts in more than one set: 0


In [9]:
# Comments per official subgroup (value >= 0.5) in each set
subgroups = pd.DataFrame({
    name: (df[MAIN_IDENTITIES] >= 0.5)[df["split"] == name].sum() for name in order
})
subgroups

,train,validation,test
male,35735,4047,4382
female,43147,4686,5188
homosexual_gay_or_lesbian,8785,1054,1071
christian,32514,3831,3888
jewish,6109,764,686
muslim,16758,1922,2017
black,11828,1445,1478
white,19625,2672,2436
psychiatric_or_mental_illness,3905,491,445


In [10]:
df = df.drop(columns="text_norm")

df.to_parquet(f"{OUT_DIR}/jigsaw_clean.parquet", index=False)
df[["id", "split"]].to_parquet(f"{OUT_DIR}/split_ids.parquet", index=False)

check = pd.read_parquet(f"{OUT_DIR}/jigsaw_clean.parquet")
print(f"Saved jigsaw_clean.parquet: {check.shape}")
print(f"Columns: {check.columns.tolist()}")

Saved jigsaw_clean.parquet: (1776349, 15)
Columns: ['id', 'comment_text', 'target', 'label', 'article_id', 'male', 'female', 'homosexual_gay_or_lesbian', 'christian', 'jewish', 'muslim', 'black', 'white', 'psychiatric_or_mental_illness', 'split']


**Note:** the seed is fixed (`SEED = 42`), so the split is always the same. `split_ids.parquet` stores the set of each comment `id` to reproduce the split.